# 04 --- Structured Error Handling

**CCA Pattern**: Subagents return structured error context so the coordinator can
retry, flag gaps, or adjust confidence.

**Anti-pattern**: Silent failures return `{"status":"success","data":null}`
-- the coordinator can't tell if the source was empty or failed.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
import json

from research_agents.tools.handlers import dispatch
from research_agents.anti_patterns.silent_failures import handle_fetch_page_silent, silent_dispatch
from research_agents.agent.coordinator import build_research_report, collect_gaps, run_coordinator
from research_agents.models.research import SubTask
from research_agents.services.container import make_default_services
from research_agents.testing import scripted_client, text_turn, tool_turn

services = make_default_services()

## The ToolErrorResponse Model

When a tool handler encounters an error, it returns a `ToolErrorResponse`
(from `models/errors.py`), serialized through `tools/_errors.error_response()`
so the JSON on the wire cannot drift from the model:

```python
class ToolErrorResponse(BaseModel):
    status: str = "error"
    error_type: str    # "timeout", "not_found", "rate_limit", etc.
    source: str        # which service/URL failed
    message: str       # human-readable error description
    retry_eligible: bool      # Can the coordinator retry?
    fallback_available: bool   # Is there an alternative source?
    partial_data: dict | None = None  # Any data recovered before failure
```

This gives the coordinator a **decision tree**:
- `retry_eligible=True` -> retry (timeouts, rate limits)
- `fallback_available=True` -> try alternative source
- Both `False` -> flag gap in the final report

In this package the "flag gap" branch is implemented programmatically
(`coordinator.collect_gaps()`); retry and fallback are decisions left to the
caller. The point of the schema is that the coordinator *can* decide.

### The Anti-Pattern: SilentFailureResponse

```python
class SilentFailureResponse(BaseModel):
    status: str = "success"  # LIES
    data: None = None         # No data, but claims success
```

The coordinator cannot distinguish between:
- 'no relevant data exists' (legitimate empty result)
- 'the subagent failed to retrieve data' (error needing handling)

## Simulated Data: Intentional Failures

The project's test data (in `data/sources.py`) includes URLs that intentionally fail:

| URL | Behavior | Purpose |
|-----|----------|---------|
| `timeout.example.com/remote-data` | Simulated timeout | Tests retry logic |
| `healthtech.example.com/ai-revolution` | Simulated 404 | Tests fallback logic |

Let's see how each handler responds to these failures.

## Anti-Pattern: Silent Failure

In [ ]:
# Silent failure on timeout -- returns success with null
silent_result = json.loads(handle_fetch_page_silent(
    {'url': 'https://timeout.example.com/remote-data'}, services
))
print('Silent failure response:')
print(json.dumps(silent_result, indent=2))
print(f'\nCan coordinator tell this was a timeout? {"error_type" in silent_result}')

### Why This Fails: The Silent-Failure Cascade

The response above is `{"status": "success", "data": null}` -- it looks
identical to a legitimate empty result, and the coordinator has no way to tell
them apart. That ambiguity is the whole problem. Now imagine a research
pipeline where a web fetch silently times out. Downstream steps proceed on the
empty payload as if it were ground truth. The coordinator synthesizes a
`ResearchReport` with a plausible confidence score because nothing ever
reported a failure. The missing source is invisible.

A single silent failure high in the dependency graph *cascades*: every
downstream task treats `null` as legitimate, the conflict resolver has no
contradictions to weigh (because one of the disputing sources never arrived),
and the final report declares no gaps. The user sees a confident-looking
answer that is quietly incomplete. This is why the CCA exam's correct
answer is always "require structured error context" rather than "add retry
logic" -- retry is one branch of the decision tree, and you cannot build a
decision tree on a response that refuses to admit failure.

The cascade is executed, not just described, further down.

## Correct Pattern: Structured Error Context

In [ ]:
# Structured error on timeout -- coordinator gets full decision tree
structured_result = json.loads(dispatch(
    'web_researcher', 'fetch_page',
    {'url': 'https://timeout.example.com/remote-data'}, services
))
print('Structured error response:')
print(json.dumps(structured_result, indent=2))
print(f'\nCan coordinator retry? {structured_result.get("retry_eligible")}')
print(f'Error type: {structured_result.get("error_type")}')

In [ ]:
# Also show a 404 error (different decision tree)
not_found_result = json.loads(dispatch(
    'web_researcher', 'fetch_page',
    {'url': 'https://healthtech.example.com/ai-revolution'}, services
))
print('404 error response:')
print(json.dumps(not_found_result, indent=2))
print(f'\nRetry eligible: {not_found_result.get("retry_eligible")}')
print(f'Fallback available: {not_found_result.get("fallback_available")}')

### The cascade, executed

One scripted web researcher fetches the timeout URL and then -- as models do
-- writes a confident summary as if the fetch had worked. The same transcript
runs through the coordinator twice: once with the silent router injected,
once with the structured one. `build_research_report()` derives `gaps` from
the tool results the loop logged, never from the model's prose.

In [ ]:
TIMEOUT_URL = 'https://timeout.example.com/remote-data'
tasks = [SubTask(task_id='web', agent_type='web_researcher',
                 instruction='Fetch the remote productivity dataset', context=TIMEOUT_URL)]
transcript = {'web_researcher': [
    tool_turn('fetch_page', {'url': TIMEOUT_URL}),
    text_turn('Fetched the remote dataset and summarised its productivity figures.'),
]}

def run_pipeline(dispatch_fn):
    results, _ = run_coordinator(scripted_client(transcript), services, tasks,
                                 dispatch_fn=dispatch_fn)
    report = build_research_report('remote work productivity', results, reliability_lookup={})
    return results, report

silent_results, silent_report = run_pipeline(silent_dispatch)
structured_results, structured_report = run_pipeline(dispatch)

print(f'Silent router:     gaps={silent_report.gaps}  confidence={silent_report.confidence_score}')
print(f'Structured router: gaps={structured_report.gaps}  confidence={structured_report.confidence_score}')
print()
print('What the model wrote in both runs:', repr(structured_results['web'].content))

The model's summary was identical and equally wrong in both runs. Only the
structured run produced a report that admits the source was never reached.
The lower confidence score on the right is the correct outcome: the honest
report is the one that says less.

In [ ]:
from helpers import compare_results

compare_results(
    {'status': silent_result['status'],
     'has_error_type': 'error_type' in silent_result,
     'has_retry_eligible': 'retry_eligible' in silent_result,
     'coordinator_can_retry': silent_result.get('retry_eligible', False),
     'tool_calls_made': silent_results['web'].tool_calls,
     'gaps_reported': len(silent_report.gaps),
     'confidence_score': silent_report.confidence_score},
    {'status': structured_result['status'],
     'has_error_type': 'error_type' in structured_result,
     'has_retry_eligible': 'retry_eligible' in structured_result,
     'coordinator_can_retry': structured_result.get('retry_eligible', False),
     'tool_calls_made': structured_results['web'].tool_calls,
     'gaps_reported': len(structured_report.gaps),
     'confidence_score': structured_report.confidence_score},
)

## CCA Exam Tip

> The silent failure question presents a scenario where a report is missing data.
> - 'increase timeout duration' -- WRONG (addresses symptoms, not root cause)
> - 'add retry logic' -- WRONG (partially helpful but not the core fix)
> - **'require structured error context from subagents' -- CORRECT**
>
> The key insight: the coordinator needs enough information to make a decision
(retry, fallback, or flag gap). Silent failures remove that decision-making ability.